# Phase 3: Financial Health Analysis
Phân tích chuyên sâu về Sức khoẻ tài chính của khách hàng. Tuân thủ nghiêm ngặt rule: **Không dùng điểm này để đánh giá tín dụng, chỉ dùng để hỗ trợ khách hàng (Non-punitive)**.
Tập trung vào các chỉ số tỷ lệ (Ratios) thay vì số tiền tuyệt đối.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import os
import sys

# Thêm src vào path để gọi visualization chuẩn màu CAM
sys.path.append(os.path.abspath(os.path.join('..')))
from src.visualization import COLORS, set_style
set_style()

DATA_DIR = os.path.join('..', 'ĐỀ BÀI', 'DATASET')
df_health = pd.read_csv(os.path.join(DATA_DIR, 'consumer_financial_health_engagement_2025.csv'))
print(f'Load thành công dữ liệu Health: {df_health.shape[0]:,} dòng')

## 3.1 Health score distribution + trends

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribution
sns.histplot(data=df_health, x='financial_health_score', bins=30, color=COLORS['primary'], ax=ax1)
ax1.set_title('Phân bố Điểm Sức khoẻ Tài chính')
ax1.set_xlabel('Financial Health Score')
ax1.set_ylabel('Số lượng bản ghi')
ax1.axvline(x=40, color=COLORS['negative'], linestyle='--', label='Ngưỡng rủi ro (<40)')
ax1.legend()

# 2. Trends by month
trend_data = df_health.groupby('analysis_month')['financial_health_score'].mean().reset_index()
sns.lineplot(data=trend_data, x='analysis_month', y='financial_health_score', 
             marker='o', color=COLORS['accent1'], linewidth=2, ax=ax2)
ax2.set_title('Xu hướng Điểm trung bình theo thời gian')
ax2.set_xlabel('Tháng')
ax2.set_ylabel('Average Score')

plt.tight_layout()
plt.show()

## 3.2 Drivers of low health (Correlation + Feature Importance)
Tìm hiểu nguyên nhân kéo tụt điểm số bằng cách tính tương quan (Correlation) và dùng mô hình Random Forest đo lường mức độ quan trọng của các biến (Feature Importance).

In [ ]:
# Chọn các biến tỷ lệ (ratios) và điểm tương tác
features = ['spend_to_income_ratio', 'credit_utilization_ratio', 'essential_spend_ratio', 
            'online_spend_ratio', 'spending_volatility', 'engagement_score']

# 1. Correlation with financial_health_score
corr = df_health[features + ['financial_health_score']].corr()['financial_health_score'].drop('financial_health_score')
corr = corr.sort_values()

plt.figure(figsize=(10, 5))
sns.barplot(x=corr.values, y=corr.index, palette=['#DC143C' if x < 0 else '#32CD32' for x in corr.values])
plt.title('Tương quan (Correlation) với Financial Health Score')
plt.xlabel('Hệ số tương quan Pearson')
plt.show()

# 2. Random Forest Feature Importance
X = df_health[features].fillna(0)
y = df_health['financial_health_score']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

fi = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_})
fi = fi.sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=fi, x='Importance', y='Feature', color=COLORS['primary'])
plt.title('Mức độ ảnh hưởng (Feature Importance) tới Health Score')
plt.show()

## 3.3 Khác biệt theo occupation, age, province

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 18))

# 1. Occupation
sns.boxplot(data=df_health, y='occupation', x='financial_health_score', color=COLORS['accent1'], ax=axes[0])
axes[0].set_title('Financial Health theo Nghề nghiệp')

# 2. Age Group
# Tạo nhóm tuổi
df_health['age_group'] = pd.cut(df_health['age'], bins=[18, 25, 35, 45, 60], labels=['18-25', '26-35', '36-45', '46-60'])
sns.boxplot(data=df_health, x='age_group', y='financial_health_score', color=COLORS['accent2'], ax=axes[1])
axes[1].set_title('Financial Health theo Nhóm tuổi')

# 3. Province (Top 10 đông nhất)
top_provinces = df_health['province_city'].value_counts().head(10).index
df_top_prov = df_health[df_health['province_city'].isin(top_provinces)]
sns.boxplot(data=df_top_prov, y='province_city', x='financial_health_score', color=COLORS['primary'], ax=axes[2])
axes[2].set_title('Financial Health theo Top 10 Tỉnh thành')

plt.tight_layout()
plt.show()

## 3.4 Crossover: Stressed × Engaged
Xác định nhóm **"Tương tác cực cao nhưng Tài chính đang gặp khó khăn"**. 
Đây là nhóm khách hàng cốt lõi cần chăm sóc (VD: Bật thông báo chi tiêu, Gợi ý công cụ quản lý ngân sách) chứ tuyệt đối không được trừng phạt (VD: khoá thẻ, giảm hạn mức).

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_health, x='engagement_score', y='financial_health_score', 
                alpha=0.5, color=COLORS['neutral'])

# Kẻ đường chia 4 góc phần tư
plt.axvline(x=60, color='gray', linestyle='--') # Ngưỡng Engaged
plt.axhline(y=50, color='gray', linestyle='--') # Ngưỡng Health

# Highlight nhóm Stressed (Health < 50) x Engaged (Engagement > 60)
stressed_engaged = df_health[(df_health['financial_health_score'] < 50) & (df_health['engagement_score'] > 60)]
sns.scatterplot(data=stressed_engaged, x='engagement_score', y='financial_health_score', 
                color=COLORS['negative'], label=f'Stressed & Engaged ({len(stressed_engaged)} rows)')

plt.title('Ma trận Crossover: Engagement vs Financial Health')
plt.xlabel('Engagement Score (Cao = Tương tác số tốt)')
plt.ylabel('Financial Health Score (Cao = Tài chính ổn định)')
plt.legend()
plt.show()

print(f"Có {len(stressed_engaged)} bản ghi tháng thuộc nhóm 'Stressed x Engaged'.")
print("Cơ hội kinh doanh: Gửi thông báo 'Bạn đã chi tiêu gần hết ngân sách' qua App (vì họ rất chăm dùng App).")